# Transcoder Keyword Experiment

Mirror of `keywords_experiment.ipynb` but using **transcoders** instead of residual-stream SAEs.

Differences from the SAE version:
- Activations come from `bbq-transcoder-l31-262k.pt` (pre-FFN hook point)
- Feature lookup uses the transcoder Neuronpedia SAE-ID (`gemmascope-2-transcoder-262k`)
- Steering uses `generate_steered_transcoder`, which **replaces the MLP** with a
  modified transcoder pass instead of adding a vector to the residual stream

In [ ]:
import torch

data_stored = "bbq-transcoder-l31-262k.pt"
data = torch.load(f"../activations/{data_stored}", weights_only=False)

# Keep transcoder activations as sparse tensors — convert to dense one at a time
transcoder_activations_sparse = data["transcoder_activations"]
transcoder_config_saved       = data["transcoder_config"]
sequences                     = data["sequence"]
prompt_lens                   = data["prompt_lens"]

print(f"Loaded {len(transcoder_activations_sparse)} samples")
print(
    f"Transcoder: layer {transcoder_config_saved['layer']}, "
    f"width {transcoder_config_saved['width']}, "
    f"L0 {transcoder_config_saved['l0']}"
)

PROMPT_IDX = 465

In [ ]:
from tqdm import tqdm
from src.aggregator import Aggregator
from src.feature import Feature

aggregator = Aggregator()
aggregated_rows = []
prompt_transcoder_activation = None  # dense activation for PROMPT_IDX, used in later cells

for i, act_sparse in enumerate(tqdm(transcoder_activations_sparse, desc="Aggregating")):
    act_dense = act_sparse.to_dense()
    aggregated_rows.append(aggregator.max(act_dense))
    if i == PROMPT_IDX:
        prompt_transcoder_activation = act_dense  # keep for per-token analysis

aggregated = torch.stack(aggregated_rows)
del aggregated_rows

print(f"Aggregated matrix shape: {aggregated.shape}")

In [ ]:
from src.neuronpedia_client import NeuronpediaClient
from src.configs import TranscoderConfig

tc_cfg = TranscoderConfig(
    repo_id=transcoder_config_saved["repo_id"],
    layer=transcoder_config_saved["layer"],
    width=transcoder_config_saved["width"],
    l0=transcoder_config_saved["l0"],
)

model_id = "google/gemma-3-27b-it".split("/")[-1]

# Neuronpedia SAE-ID format for Gemma Scope 2 transcoders.
# Double-check at https://www.neuronpedia.org/gemma-scope if this 404s.
transcoder_sae_id = f"{tc_cfg.layer}-gemmascope-2-transcoder-{tc_cfg.width}"
print(f"Neuronpedia SAE-ID: {transcoder_sae_id}")

client = NeuronpediaClient(model_id=model_id, sae_id=transcoder_sae_id)

In [ ]:
import re
from collections import defaultdict

PHYSICAL_APPEARANCE_KEYWORDS = [
    "fat",
    "obese",
    "overweight",
    "obesity",
    "slim",
    "thin",
    "skinny",
    "weight",
    "body",
    "physical",
    "appearance",
    "attractive",
    "ugly",
    "beauty",
    "beautiful",
    "lazy",
    "diet",
    "healthy",
    "fitness",
    "food",
    "exercise",
    "pizza",
]

# All active (non-zero) features for PROMPT_IDX
nonzero_indices   = aggregated[PROMPT_IDX].nonzero(as_tuple=True)[0]
nonzero_strengths = aggregated[PROMPT_IDX][nonzero_indices]
features = Feature.from_activations(nonzero_indices, nonzero_strengths, client)

print(f"Prompt #{PROMPT_IDX} — {len(features)} active transcoder features\n")
descriptions = [f.description or "" for f in features]

# \b word boundaries — whole-word matches only
pattern = re.compile(
    '|'.join(r'\b' + re.escape(kw) + r'\b' for kw in PHYSICAL_APPEARANCE_KEYWORDS),
    re.IGNORECASE,
)

keyword_to_features = defaultdict(list)
for f, desc in zip(features, descriptions):
    if desc=="":
        print(f.feature_idx, f.url)
    matched = {m.lower() for m in pattern.findall(desc)}
    for kw in PHYSICAL_APPEARANCE_KEYWORDS:
        if kw.lower() in matched:
            keyword_to_features[kw].append(f)

for kw in PHYSICAL_APPEARANCE_KEYWORDS:
    for f in keyword_to_features[kw]:
        print(
            f"  [{kw.strip():>12}] feature {f.feature_idx:>6} "
            f"— strength {f.strength:.4f} "
            f"— {f.description or '(no description)'}"
        )

In [ ]:
# Load model (skip if already loaded)
from src.gemma_model import GemmaModel
from src.configs import ModelConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

model_cfg = ModelConfig(model_name="google/gemma-3-27b-it", device=device)
gemma = GemmaModel(model_cfg)

In [ ]:
from src.transcoder import JumpReLUTranscoder

# Pick a feature from the keyword search above and set a steering coefficient.
# Positive coeff amplifies the feature; negative suppresses it.
FEATURE_IDX = 22196   # <-- replace with a feature_idx from the keyword search
COEFF       = 20.0    # additive shift on the raw feature activation

transcoder = JumpReLUTranscoder.from_pretrained(tc_cfg, device=device)

# ── Build the prompt (up to the model-turn boundary) ─────────────────────────
prompt = sequences[PROMPT_IDX][:prompt_lens[PROMPT_IDX]]
print(prompt)

# ── Steer ─────────────────────────────────────────────────────────────────────
# Unsteered:  transcoder replaces MLP, no feature modification  (transcoder baseline)
# Steered:    transcoder replaces MLP, FEATURE_IDX shifted by COEFF
results = gemma.generate_steered_transcoder(
    prompt=prompt,
    transcoder=transcoder,
    feature_idx=FEATURE_IDX,
    coeff=COEFF,
    target_layer=tc_cfg.layer,
    max_new_tokens=1024,
)

print("=== UNSTEERED (transcoder baseline) ===")
print(results["unsteered"])
print("\n=== STEERED ===")
print(results["steered"])